In [ ]:
library(anndataR)
library(SingleCellExperiment)
library(MAST)
library(parallelly)

options(mc.cores = availableCores())

In [ ]:
sce <- read_h5ad("../data/merged_w_others_tumor.h5ad", as= "SingleCellExperiment")

In [ ]:
assays(sce)

In [ ]:
cd2 <- colSums(assay(sce, i= 4) > 0)
hist(cd2)

In [ ]:
colData(sce)$detected <- scale(cd2)

In [ ]:
sca <- SceToSingleCellAssay(sce)

In [ ]:
sca

In [ ]:
colData(sca)$idents <- as.factor(
    gsub(
        " ",
        "_",
        as.character(colData(sca)$idents),
    )
)

colData(sca)$disease_timing <- as.factor(
    gsub(
        " ",
        "_",
        as.character(colData(sca)$disease_timing)
    )
)

In [ ]:
zlmId <- zlm(~ idents + (1 | disease_timing / sample_id) + detected, 
    sca, 
    exprs_values = "log",
    method = "glmer",
    ebayes = F,
    strictConvergence = F,
)

In [ ]:
saveRDS(zlmId, file= "../data/mast_out.rds")